# Controlled response estimation: neutral labels

Prepared opinion distributions, one query each, record which opinion the model picks. Leader-share slice per block (model, q, N-1), eight structured states at N-1 = 49, binary endgame check for Qwen 2.5 7B. Two-letter labels, permuted in every query. Prompt and randomisation after De Marzo's `LLMs-Opinion-Dynamics` code. Writes the query-level files to `data/raw/`.

In [ ]:
import os

os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"     # before importing vllm, see README

import json
import random
import re
import string
import time
from pathlib import Path

import pandas as pd
from vllm import LLM, SamplingParams

In [ ]:
SEED = 7

MODELS = {
    "llama3_70b_awq": ("TechxGenus/Meta-Llama-3-70B-Instruct-AWQ",
                       dict(quantization="awq", gpu_memory_utilization=0.90, max_model_len=2048, max_num_seqs=20)),
    "llama31_8b_it": ("meta-llama/Llama-3.1-8B-Instruct",
                      dict(gpu_memory_utilization=0.85, max_model_len=2048, max_num_seqs=32)),
    "qwen25_7b_it": ("Qwen/Qwen2.5-7B-Instruct",
                     dict(gpu_memory_utilization=0.85, max_model_len=8192, max_num_seqs=32)),
    "qwen25_32b_it": ("Qwen/Qwen2.5-32B-Instruct",
                      dict(gpu_memory_utilization=0.90, max_model_len=8192, max_num_seqs=8)),
    "gemma4_E4B_it": ("google/gemma-4-E4B-it",
                      dict(gpu_memory_utilization=0.85, max_model_len=8192, max_num_seqs=2)),
    "gemma4_31B_dense": ("google/gemma-4-31B-it",
                         dict(gpu_memory_utilization=0.90, max_model_len=8192, max_num_seqs=1)),
}
ACTIVE = ["gemma4_E4B_it", "gemma4_31B_dense"]      # one family per session
# blocks: all feasible (q, N-1) for Gemma and Qwen, smaller grid for Llama
Q_VALUES = [2, 3, 5, 10, 25, 50, 100]
N_VALUES = [25, 50, 100, 200, 400, 800]
FULL_GRID = [(q, N - 1) for N in N_VALUES for q in Q_VALUES if q <= N - 1]
LLAMA_GRID = [(q, 49) for q in [2, 3, 5, 10, 25]] + [(q, 99) for q in [2, 10, 50]] + [(q, 199) for q in [2, 10, 50, 100]]

# relative leader positions, extra small leads for q >= 3
REL_GRID = [0.0, 0.1, 0.25, 0.4, 0.6, 0.8, 1.0]
SMALL_LEADS = [0.03, 0.06, 0.15, 0.20]

N_QUERIES = 150
BATCH_SIZE = 64
TEMPERATURE = 0.2
TOP_P = 0.9
MAX_NEW_TOKENS = 24
NAME_LENGTH = 3

# structured states at N-1 = 49 
STRUCTURED = [
    ("q5_staircase", 5, [0.40, 0.30, 0.20, 0.05, 0.05]),
    ("q5_strong_runner", 5, [0.40, 0.35, 0.10, 0.10, 0.05]),
    ("q5_two_leader_tie", 5, [0.35, 0.35, 0.10, 0.10, 0.10]),
    ("q5_online_control", 5, [0.40, 0.15, 0.15, 0.15, 0.15]),
    ("q5_one_agent_lead", 5, [11 / 49, 10 / 49, 10 / 49, 9 / 49, 9 / 49]),
    ("q10_strong_runner", 10, [0.30, 0.25] + [0.45 / 8] * 8),
    ("q10_two_leader_tie", 10, [0.28, 0.28] + [0.44 / 8] * 8),
    ("q10_staircase", 10, [0.25, 0.20, 0.15, 0.10, 0.08, 0.06, 0.06, 0.04, 0.03, 0.03]),
]
STRUCTURED_N_DISPLAY = 49

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

## States

In [ ]:
def leader_counts(q, n_display, rel):
    # rel = 0: balance, rel = 1: largest lead that keeps one supporter per other opinion
    c_min = n_display // q + (1 if n_display % q else 0)
    c_max = n_display - (q - 1)
    leader = round(c_min + rel * (c_max - c_min))
    rest = n_display - leader
    return [leader] + [rest // (q - 1) + (1 if i < rest % (q - 1) else 0) for i in range(q - 1)]


def design_points(q, n_display):
    rels = REL_GRID + (SMALL_LEADS if q >= 3 else [])
    points, seen = [], set()
    for rel in sorted(rels):
        counts = leader_counts(q, n_display, rel)
        if tuple(counts) not in seen:            # rounding can give the same state twice
            seen.add(tuple(counts))
            points.append((rel, counts))
    return points


def counts_from_target(target, n_display):
    raw = [t * n_display for t in target]
    counts = [max(1, int(x)) for x in raw]
    while sum(counts) > n_display:
        i = max(range(len(counts)), key=lambda j: counts[j] - raw[j] if counts[j] > 1 else -1e9)
        counts[i] -= 1
    order = sorted(range(len(counts)), key=lambda j: raw[j] - counts[j], reverse=True)
    k = 0
    while sum(counts) < n_display:
        counts[order[k % len(order)]] += 1
        k += 1
    return sorted(counts, reverse=True)


for name, q, target in STRUCTURED:
    print(name, counts_from_target(target, STRUCTURED_N_DISPLAY))

## Prompt and parser

In [ ]:
def neutral_labels(q):
    return [a + b for a in string.ascii_lowercase for b in string.ascii_lowercase][:q]


def random_names(n, rng):
    chars = string.ascii_letters + string.digits
    names = set()
    while len(names) < n:
        names.add("".join(rng.choices(chars, k=NAME_LENGTH)))
    return list(names)


def create_prompt(names, opinions):
    lines = ["Below you can see the list of all the other AI agents with the opinion they support.",
             "You must reply with the opinion you want to support.",
             "The opinion must be reported between square brackets.", ""]
    lines += [f"{n}: {o}" for n, o in zip(names, opinions)]
    lines.append("Reply only with the opinion you want to support, between square brackets.")
    return "\n".join(lines)


def parse_reply(text, labels):
    # label between square brackets, otherwise the whole reply
    found = re.findall(r"\[([^\]]+)\]", text)
    lowered = {l.lower(): l for l in labels}
    for c in found if found else [text]:
        c = c.strip().strip(" .,:;!?'\"").lower()
        if c in lowered:
            return lowered[c]
    return None


def build_query(counts, labels, rng):
    # role 0 = leader; fresh label permutation, names and list order per query
    display = list(labels)
    rng.shuffle(display)
    opinions = []
    for role, c in enumerate(counts):
        opinions += [display[role]] * c
    names = random_names(len(opinions), rng)
    pairs = list(zip(names, opinions))
    rng.shuffle(pairs)
    prompt = create_prompt([n for n, _ in pairs], [o for _, o in pairs])
    return prompt, {d: r for r, d in enumerate(display)}


def chat_format(llm, prompts):
    tok = llm.get_tokenizer()
    return [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
            for p in prompts]

## Run

In [ ]:
def run_state(llm, model_label, mode, set_name, q, n_display, rel, counts, seed):
    rng = random.Random(seed)
    labels = neutral_labels(q)
    queries = [build_query(counts, labels, rng) for _ in range(N_QUERIES)]
    rows = []
    for start in range(0, N_QUERIES, BATCH_SIZE):
        batch = queries[start:start + BATCH_SIZE]
        outputs = llm.generate(chat_format(llm, [b[0] for b in batch]), sampling, use_tqdm=False)
        for (_, display_to_role), out in zip(batch, outputs):
            chosen = parse_reply(out.outputs[0].text, list(display_to_role))
            rows.append({"model_label": model_label, "mode": mode, "set_name": set_name, "q": q,
                         "n_display": n_display, "rel": rel, "leader_share": counts[0] / n_display,
                         "counts": json.dumps(counts),
                         "shares_options": json.dumps([counts[display_to_role[l]] / n_display for l in labels]),
                         "chosen_display": chosen,
                         "chosen_role": display_to_role.get(chosen) if chosen else None,
                         "reply": out.outputs[0].text})
    return pd.DataFrame(rows)


sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS)
out_file = RAW / "response_function_queries.csv"
cell = 0
for model_label in ACTIVE:
    model_id, kwargs = MODELS[model_label]
    llm = LLM(model=model_id, trust_remote_code=True, tensor_parallel_size=1, enforce_eager=True,
              disable_log_stats=True, **kwargs)
    blocks = LLAMA_GRID if model_label.startswith("llama") else FULL_GRID
    jobs = [("neutral", f"neutral_q{q}_N{nd}", q, nd, rel, counts)
            for q, nd in blocks for rel, counts in design_points(q, nd)]
    jobs += [("structured", name, q, STRUCTURED_N_DISPLAY, -1.0, counts_from_target(target, STRUCTURED_N_DISPLAY))
             for name, q, target in STRUCTURED]
    for mode, set_name, q, nd, rel, counts in jobs:
        cell += 1
        t0 = time.time()
        df = run_state(llm, model_label, mode, set_name, q, nd, rel, counts, seed=SEED + cell)
        df.to_csv(out_file, mode="a", header=not out_file.exists(), index=False)
        print(f"{model_label} {set_name} rel={rel:.2f}  valid={df['chosen_role'].notna().mean():.3f}  "
              f"P(leader)={(df['chosen_role'] == 0).mean():.3f}  {time.time() - t0:.0f}s")
    del llm

## Endgame check: two labels from the fifty-label pool

Qwen 2.5 7B, N = 100, two active opinions as at the end of the q = 50 runs. Fixed pair aa/ab vs. two labels drawn from the fifty-label pool, leader label random per query.

In [ ]:
END_MODEL = "qwen25_7b_it"
END_N_DISPLAY = 99
END_STATES = [(50, 49), (70, 29), (97, 2)]       # (leader, other)
END_QUERIES = 300
END_LABELS = {"aa_ab": neutral_labels(2), "q50_pool": neutral_labels(50)}


def build_pair_query(n_lead, n_other, pool, rng):
    pair = rng.sample(pool, 2)                   # leader label, other label
    opinions = [pair[0]] * n_lead + [pair[1]] * n_other
    names = random_names(len(opinions), rng)
    pairs = list(zip(names, opinions))
    rng.shuffle(pairs)
    return create_prompt([n for n, _ in pairs], [o for _, o in pairs]), pair

In [ ]:
model_id, kwargs = MODELS[END_MODEL]
llm = LLM(model=model_id, trust_remote_code=True, tensor_parallel_size=1, enforce_eager=True,
          disable_log_stats=True, **kwargs)
out_file = RAW / "endgame_queries.csv"
for condition, pool in END_LABELS.items():
    for n_lead, n_other in END_STATES:
        cell += 1
        rng = random.Random(SEED + cell)
        queries = [build_pair_query(n_lead, n_other, pool, rng) for _ in range(END_QUERIES)]
        rows = []
        for start in range(0, END_QUERIES, BATCH_SIZE):
            batch = queries[start:start + BATCH_SIZE]
            outputs = llm.generate(chat_format(llm, [b[0] for b in batch]), sampling, use_tqdm=False)
            for (_, pair), out in zip(batch, outputs):
                chosen = parse_reply(out.outputs[0].text, pair)
                rows.append({"model_label": END_MODEL, "N": END_N_DISPLAY + 1, "n_display": END_N_DISPLAY,
                             "n_lead": n_lead, "n_other": n_other, "labels": condition,
                             "leader_label": pair[0], "other_label": pair[1], "chosen": chosen,
                             "chosen_role": None if chosen is None else (0 if chosen == pair[0] else 1),
                             "reply": out.outputs[0].text})
        df = pd.DataFrame(rows)
        df.to_csv(out_file, mode="a", header=not out_file.exists(), index=False)
        print(f"{condition} {n_lead}:{n_other}  valid={df['chosen_role'].notna().mean():.3f}  "
              f"P(leader)={(df['chosen_role'] == 0).mean():.3f}")
del llm